# Generación de Texto con modelos GPT

Version Ivan Moran y Erik Vergara

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ohtar10/icesi-nlp/blob/main/Sesion5/1-text-generation.ipynb)

En este notebook harémos uso de un modelo tipo GPT-2 pre-entrenado en idioma español que utilizaremos para generar texto a partir de un contexto inicial que proveerémos. Luego, harémos fine tuning a este modelo con un dataset de chistes en español y observar como cambia la generación de texto en función del dataset que utilicemos.

#### Referencias
- Dataset: https://huggingface.co/datasets/mrm8488/CHISTES_spanish_jokes
- [Improving Language Understanding by Generative Pre-Training](https://cdn.openai.com/research-covers/language-unsupervised/language_understanding_paper.pdf)
- [Natural Language Processing with Transformers: Building Language Applications With Hugging Face](https://www.amazon.com/Natural-Language-Processing-Transformers-Applications/dp/1098103246)
- [GPT2 Spanish](https://huggingface.co/DeepESP/gpt2-spanish)
- [Fine-Tune a non-Englush GPT-2 Model with Huggingface](https://www.philschmid.de/fine-tune-a-non-english-gpt-2-model-with-huggingface)

# Paquetes y dependencias

Se mantiene la instalacion de paquetes y dependencias.

In [1]:
import pkg_resources
import warnings

warnings.filterwarnings('ignore')

installed_packages = [package.key for package in pkg_resources.working_set]
IN_COLAB = 'google-colab' in installed_packages

/tmp/ipykernel_9416/2396000874.py:1: DeprecationWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html
  import pkg_resources


In [2]:
#!test '{IN_COLAB}' = 'True' && wget  https://github.com/Ohtar10/icesi-nlp/raw/refs/heads/main/requirements.txt && pip install -r requirements.txt
!test '{IN_COLAB}' = 'True' && sudo apt-get update -y
!test '{IN_COLAB}' = 'True' && sudo apt-get install python3.10 python3.10-distutils python3.10-lib2to3 -y
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.11 2
!test '{IN_COLAB}' = 'True' && sudo update-alternatives --install /usr/local/bin/python python /usr/bin/python3.10 1
!test '{IN_COLAB}' = 'True' && pip install lightning datasets 'transformers[torch]' sentence-transformers torchinfo evaluate

Get:1 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:2 https://cli.github.com/packages stable InRelease [3,917 B]
Get:3 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:4 https://cli.github.com/packages stable/main amd64 Packages [354 B]
Get:5 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:6 https://r2u.stat.illinois.edu/ubuntu jammy InRelease [6,555 B]
Hit:7 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  Packages [2,473 kB]
Get:9 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:10 https://r2u.stat.illinois.edu/ubuntu jammy/main all Packages [9,932 kB]
Hit:11 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease
Hit:12 https://ppa.launchpadcontent.net/graphics-drivers/ppa/ubuntu jammy InRelease
Hit:13 https://ppa.launchpadcontent.net/ubuntugis/ppa/ubuntu j

# Comparacion de modelos en la generacion de contenido

A continuacion vamos a definir varios modelos para utilizarlos en la generacion de contenido, en este caso, creacion narrativa tipo cuentos, aprovechando el corpus con el que fue entrenado el modelo base DeepESP/gpt2-spanish.

1. DeepESP/gpt2-spanish
2. meta-llama/Llama-3.2-1B
3. API meta-llama/Llama-3.2-1B-Instruct:novita

## 1. Generative pre-training Transformer - GPT

Se mantiene un modelo GPT para poder comparar la generacion de contenido. Sin embargo, es importante tener en cuenta que este modelo (DeepESP/gpt2-spanish) fue entrenado con un corpus de 11,5 GB de textos, que corresponden a 3,5 GB de artículos de Wikipedia y 8 GB de libros (narrativa, cuentos, teatro, poesía, ensayos y divulgación).

### Definicion del modelo

In [3]:
import torch
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

device = "cuda" if torch.cuda.is_available() else "cpu"
model_name = "DeepESP/gpt2-spanish"
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModelForCausalLM.from_pretrained(model_name).to(device)
model

config.json:   0%|          | 0.00/914 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/115 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/262 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/261M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/261M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/149 [00:00<?, ?it/s]

The tied weights mapping and config for this model specifies to tie transformer.wte.weight to lm_head.weight, but both are present in the checkpoints, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning
GPT2LMHeadModel LOAD REPORT from: DeepESP/gpt2-spanish
Key                                     | Status     |  | 
----------------------------------------+------------+--+-
transformer.h.{0...11}.attn.bias        | UNEXPECTED |  | 
transformer.h.{0...11}.attn.masked_bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


GPT2LMHeadModel(
  (transformer): GPT2Model(
    (wte): Embedding(50257, 768)
    (wpe): Embedding(1024, 768)
    (drop): Dropout(p=0.1, inplace=False)
    (h): ModuleList(
      (0-11): 12 x GPT2Block(
        (ln_1): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (attn): GPT2Attention(
          (c_attn): Conv1D(nf=2304, nx=768)
          (c_proj): Conv1D(nf=768, nx=768)
          (attn_dropout): Dropout(p=0.1, inplace=False)
          (resid_dropout): Dropout(p=0.1, inplace=False)
        )
        (ln_2): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
        (mlp): GPT2MLP(
          (c_fc): Conv1D(nf=3072, nx=768)
          (c_proj): Conv1D(nf=768, nx=3072)
          (act): NewGELUActivation()
          (dropout): Dropout(p=0.1, inplace=False)
        )
      )
    )
    (ln_f): LayerNorm((768,), eps=1e-05, elementwise_affine=True)
  )
  (lm_head): Linear(in_features=768, out_features=50257, bias=False)
)

### Generando texto

Utilizamos el principio del modelo de Completitud estadística, para predecir el siguiente token basándose el corpus literario con el que fue entrenado, para completar una historia, en donde le damos el inicio del cuento.

In [4]:
# 1. Definimos el Prompt con un ejemplo previo para marcar el estilo narrativo
prompt_cuento = """[Estilo: Narrativa fantástica, descriptiva y emocional]

Ejemplo de leyenda breve:
El viejo relojero no reparaba el tiempo, sino los recuerdos. Con un destornillador de plata y una gota de nostalgia, lograba que los péndulos volvieran a marcar el instante exacto en que alguien fue feliz. El taller olía a madera antigua y a momentos recuperados.

Historia corta:
Título: El secreto de la montaña de cristal.
Había una vez en el corazón de un valle olvidado, donde las nubes rozan el suelo y el viento susurra secretos antiguos, un joven buscador de tesoros que"""

inputs = tokenizer(prompt_cuento, return_tensors="pt").to(device)

# 2. Generación optimizada para narrativa
output = model.generate(
    input_ids=inputs["input_ids"],
    attention_mask=inputs["attention_mask"],
    max_new_tokens=100,          # Damos más espacio para que la historia se desarrolle
    do_sample=True,
    temperature=0.9,             # Un poco más bajo que en poesía para mantener la lógica lineal
    top_k=50,
    top_p=0.95,
    repetition_penalty=1.5,      # Evita que repita "Había una vez" o nombres de personajes
    no_repeat_ngram_size=4,      # Evita que repita frases de 3 palabras (clave en cuentos)
    pad_token_id=tokenizer.eos_token_id
)

# 3. Resultado
texto_final = tokenizer.decode(output[0], skip_special_tokens=True)
print(texto_final)

[Estilo: Narrativa fantástica, descriptiva y emocional]

Ejemplo de leyenda breve:
El viejo relojero no reparaba el tiempo, sino los recuerdos. Con un destornillador de plata y una gota de nostalgia, lograba que los péndulos volvieran a marcar el instante exacto en que alguien fue feliz. El taller olía a madera antigua y a momentos recuperados.

Historia corta:
Título: El secreto de la montaña de cristal.
Había una vez en el corazón de un valle olvidado, donde las nubes rozan el suelo y el viento susurra secretos antiguos, un joven buscador de tesoros que había descubierto su propio camino hasta alcanzar sus propias profundidades (el mundo subterráneo). 

Las historias cortas fueron muy útiles para entender lo largo del texto. No era algo con sentido; eran cosas escritas por mentes humanas o muertas vivas por personas vivas u muertos vivientes… pero también constituían parte del relato de cómo se convirtió aquel hombre extraordinario: lo tomó prestado y lo guardó como si fuese suyo al 

**Observacion:**

- Se observa que la narrativa no es tan coherente, a pesar de que el corpus con el que fue entrenado tiene mayoritariamente contenido narrativo, cuentos, poemas, entre otros.

## 2. Llama 3.2 collection of multilingual large language models (LLMs)

Modelos generativos preentrenados y optimizados para instrucciones en tamaños de 1B y 3B (texto de entrada/texto de salida). Los modelos Llama 3.2, optimizados para texto, están diseñados para casos de uso de diálogo multilingüe, incluyendo tareas de recuperación y resumen de información. Superan a muchos de los modelos de chat de código abierto y propietarios disponibles en los benchmarks habituales del sector.



### Conectarse con Hugging Face

Primero se debe crear una conexion con Hugging Face para descargar el modelo }(Es libre pero se debe autenticar para utilizarlo), sirve tambien para consumir el modelo en fomra de servicio.


In [5]:
import os
from huggingface_hub import login
import getpass

# Replace 'YOUR_HF_TOKEN_HERE' with your actual Hugging Face token
os.environ['HF_TOKEN'] = 'YOUR_HF_TOKEN_HERE'

print("HF_TOKEN environment variable has been set for this session.")

HF_TOKEN environment variable has been set for this session.


### Definicion del modelo

In [6]:
import torch
from transformers import pipeline

# 1. Configuración del modelo Llama 3.2 (1B es ideal para velocidad)
model_id = "meta-llama/Llama-3.2-1B"

pipe = pipeline(
    "text-generation",
    model=model_id,
    torch_dtype=torch.bfloat16,
    device_map="auto"
)

config.json:   0%|          | 0.00/843 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/146 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/185 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/301 [00:00<?, ?B/s]

### Generando texto

In [7]:
# 2. Definición del Prompt Elaborado
# Usamos una estructura que Llama reconoce: Instrucción + Estilo + Inicio
prompt_cuento = """Escribe un cuento corto siguiendo este estilo: Narrativa fantástica, descriptiva y emocional.

Historia:
Título: El secreto de la montaña de cristal.
Había una vez en el corazón de un valle olvidado, donde las nubes rozan el suelo y el viento susurra secretos antiguos, un joven buscador de tesoros que"""

# 3. Generación con parámetros optimizados para Llama
# Nota: Llama 3.2 funciona mejor con una temperatura ligeramente más baja para no perder el hilo
resultados = pipe(
    prompt_cuento,
    max_new_tokens=100,      # Llama puede escribir historias más largas con coherencia
    do_sample=True,
    temperature=0.7,
    top_p=0.9,
    repetition_penalty=1.2,  # Llama es más inteligente, no necesita una penalización tan agresiva como GPT-2
    eos_token_id=pipe.tokenizer.eos_token_id,
    pad_token_id=pipe.tokenizer.eos_token_id
)

# 4. Mostrar el resultado
print(resultados[0]['generated_text'])

Passing `generation_config` together with generation-related arguments=({'temperature', 'top_p', 'max_new_tokens', 'repetition_penalty', 'do_sample', 'eos_token_id', 'pad_token_id'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
Both `max_new_tokens` (=100) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Escribe un cuento corto siguiendo este estilo: Narrativa fantástica, descriptiva y emocional.

Historia:
Título: El secreto de la montaña de cristal.
Había una vez en el corazón de un valle olvidado, donde las nubes rozan el suelo y el viento susurra secretos antiguos, un joven buscador de tesoros que se llama Hombre del bosque. Un día llegó a esa tierra para encontrar los restos perdidos de un viejo reino llamados los Pecadores. Se ha perdido toda memoria sobre ellos pero sí había pasado entre ellas una historia muy extraña con misteriosas reliquias. 
En aquellos días existían dos especies humanas distintivas; uno era blanco como el sol y otro negro como la noche. La primera fue creada por Dios cuando


**Observacion:**
- Se observa una mejoria en la creacion narrativa, aunque le falta continuidad en la historia.

## 3. API Llama 3.2 de modelos multilingües de lenguaje grande (LLM)

Se realizó una prueba de inferencia remota utilizando el Hugging Face Router API para consumir el modelo Llama-3.2-1B-Instruct. A diferencia de la ejecución local, este enfoque emplea una arquitectura de chat (vía el endpoint de Novita) que permite enviar un prompt estructurado mediante roles de sistema y usuario, optimizando la generación de texto mediante el procesamiento en la nube sin carga directa en el hardware local.

In [8]:
import os
from openai import OpenAI

# 1. Configuración del cliente (Asegúrate de tener HF_TOKEN en tus variables de entorno)
client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=os.environ["HF_TOKEN"],
)

# 2. Llamada al modelo Llama 3.2 vía API
completion = client.chat.completions.create(
    model="meta-llama/Llama-3.2-1B-Instruct:novita",
    messages=[
        {
            "role": "system",
            "content": "Eres un escritor creativo experto en narrativa fantástica, descriptiva y emocional."
        },
        {
            "role": "user",
            "content": "Escribe un cuento corto titulado 'El secreto de la montaña de cristal'. La historia debe comenzar en el corazón de un valle olvidado donde las nubes rozan el suelo."
        }
    ],
    temperature=0.7,
    max_tokens=500
)

# 3. Impresión del resultado para comparar
print(completion.choices[0].message.content)

**El secreto de la montaña de cristal**

En las profundidades de un valle olvidado, rodeado de nubes de nieve que parecían yergir el cielo, existía una montaña de cristal. La montaña de cristal era un lugar de leyendas y misterios, donde la naturaleza expresaba su magia y su poder.

La historia comienza en el pequeño pueblo de Los Lagos, donde la gente vivía en silencio y miedo. La montaña de cristal era un lugar de sueños, un lugar donde los niños veían fantasmas y las mujeres venían a pedir el bienestar de sus hijos. Pero la realidad era diferente.

La montaña de cristal era un lugar donde las piedras parecían hablar, donde las hojas del arbusto parecían vestirse de colores y donde el viento parecía cantar canciones misteriosas. Era un lugar donde la naturaleza era el maestro y el humanismo era solo un recuerdo lejano.

Una noche, una joven llamada Sofía decidió investigar la historia de la montaña de cristal. Se despertó temblorosa y se puso de pie en el pequeño pueblo, con la inten

**Observaciones:**
- El modelo entiende mejor la instruccion y el cuento tiene coherencia narrativa y continuidad en su historia

# Conclusiones

- El modelo GPT sufrió de deriva de contexto. En lugar de usar el ejemplo del que se le proporciono en el prompt como una guía de estilo, absorbió el contenido semántico y empezó a divagar en el contenido.
- El modelo Llama 3.2 continuó el texto a partir del contenido ingresado en el prompt. Sin embargo, la narrativa le hace falta continuidad y creatividad.
- El modelo Llama 3.2, consumido por API, entiende mejor las instrucciones e interpretó como una orden de escritura creativa y redactó un relato estructurado, con mejor narrativa que los otros modelos.